# MT3 — YOLO26 Entity Detector + Edge Benchmark

Trains the **YOLO26** entity detector with the *same recipe* as the frozen YOLOv11s baseline (fair
one-thing-changed swap), then measures the **MT3 edge metrics**. Reuses the same dataset zip.

**DISCONNECT-SAFE:** training output is written straight to Google Drive (`unsafe-net-yolo26-out/runs26`),
so `best.pt` persists continuously even if the Colab runtime dies mid-run — no reliance on a final save cell.

**Runtime → GPU before running.** Kaggle works too (30 h/wk free GPU): swap the Drive paths for
`/kaggle/working/...` and upload the dataset as a Kaggle Dataset. For real Jetson numbers, run cell 7 on the Jetson.


## 1. GPU


In [ ]:
!nvidia-smi


## 2. Install Ultralytics (latest — includes YOLO26)


In [ ]:
!pip -q install -U ultralytics
import torch, ultralytics; from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 3. Mount Drive + dataset + persistent output dir


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, glob
OUTDIR='/content/drive/MyDrive/unsafe-net-yolo26-out'; os.makedirs(OUTDIR, exist_ok=True)  # weights land here, survive disconnect
ZIP='/content/drive/MyDrive/unsafe-net-detector-dataset.zip'
assert os.path.exists(ZIP), 'Upload unsafe-net-detector-dataset.zip to My Drive root (same as YOLO11 run).'
!cp "$ZIP" /content/ds.zip && rm -rf /content/ds && mkdir -p /content/ds && unzip -q /content/ds.zip -d /content/ds
print('train', len(glob.glob('/content/ds/images/train/*.jpg')), '| val', len(glob.glob('/content/ds/images/val/*.jpg')), '| out', OUTDIR)


## 4. data.yaml (nc=8, identical to YOLO11 run)


In [ ]:
open('/content/ds/data.yaml','w').write('''path: /content/ds\ntrain: images/train\nval: images/val\nnc: 8\nnames:\n  0: Person_Authorized\n  1: Person_Unauthorized\n  2: Safe_Walkway\n  3: Restricted_Zone\n  4: Panel_Closed\n  5: Panel_Opened\n  6: Forklift_Safe_Load\n  7: Forklift_Overload\n''')
print(open('/content/ds/data.yaml').read())


## 5. Train YOLO26 — SAME recipe as YOLOv11s baseline (fair swap)
2-stage freeze→fine-tune, SGD lr 0.01→0.001, imgsz 800, hsv_h=0 (protect vest colour), mosaic 0.5, seed 0.
**Output goes to Drive** (`$OUTDIR/runs26`) so best.pt is saved continuously as training improves.


In [ ]:
RUNS=OUTDIR+'/runs26'   # <-- Drive-backed project dir: disconnect-safe
AUG=dict(hsv_h=0.0,hsv_s=0.5,hsv_v=0.4,degrees=0.0,flipud=0.0,fliplr=0.5,translate=0.1,scale=0.5,mosaic=0.5,close_mosaic=10,mixup=0.0)
COMMON=dict(data='/content/ds/data.yaml',imgsz=800,batch=16,device=0,optimizer='SGD',momentum=0.937,weight_decay=0.0005,seed=0,patience=30,plots=True,project=RUNS,exist_ok=True,**AUG)
m=YOLO('yolo26s.pt')   # if unavailable, fall back to 'yolo11s.pt' and note the version
m.train(name='stage1_frozen',freeze=10,epochs=30,lr0=0.01,**COMMON)
m2=YOLO(RUNS+'/stage1_frozen/weights/best.pt')
m2.train(name='stage2_finetune',freeze=0,epochs=100,lr0=0.001,**COMMON)
BEST=RUNS+'/stage2_finetune/weights/best.pt'
import shutil; shutil.copy(BEST, OUTDIR+'/yolo26_best.pt')   # canonical name for local download
print('best:',BEST,'\nDOWNLOAD THIS ->', OUTDIR+'/yolo26_best.pt')


## 6. Validate — compare to YOLO11 baseline (mAP@50 0.969 / 50-95 0.894)


In [ ]:
best=YOLO(BEST); mm=best.val(data='/content/ds/data.yaml',imgsz=800,device=0)
print('YOLO26 mAP50=',round(float(mm.box.map50),3),' mAP50-95=',round(float(mm.box.map),3),' (YOLO11 was 0.969 / 0.894)')
for i,c in enumerate(mm.box.ap_class_index): print(f'  {best.names[int(c)]:20s} mAP50={mm.box.ap50[i]:.3f} R={mm.box.r[i]:.3f}')


## 7. MT3 edge metrics — params/FLOPs, size, per-format latency, INT8
`benchmark()` runs PyTorch/ONNX/TensorRT(FP16/INT8) and reports size + inference ms + mAP.
On Colab GPU = proxy; **run this cell on the Jetson for the real MT3 numbers.**
Copy the printed summary table into the report (TensorRT FP16/INT8 rows matter most).


In [ ]:
best.info()   # params + GFLOPs
from ultralytics.utils.benchmarks import benchmark
df = benchmark(model=BEST, data='/content/ds/data.yaml', imgsz=800, half=True, int8=True)
print(df)   # <- paste this table into the report
df.to_csv(OUTDIR+'/yolo26_benchmark.csv')   # also saved to Drive


## Next (local)
Download `yolo26_best.pt` from `My Drive/unsafe-net-yolo26-out/` → drop at
`sweep_results/entity_detector/yolo26_best.pt`. Then I re-extract tokens
(`build_relation_tokens.py --detector sweep_results/entity_detector/yolo26_best.pt --appearance
--appearance_backbone yolo`), re-run the frozen S1a pipeline, and compare vs the YOLO11 baseline
(`logits_yoloroi.pt`) with paired McNemar + LCO scene-robust.

**Jetson:** run cell 7 (or export+benchmark using yolo26_best.pt) on the device for real
latency p50/p95/p99, FPS, multi-stream, INT8 size, and energy (J/inference, FPS/W).
